# v43 — лучший checkpoint v42 против рабочего v25

Run All в основном проекте, в той же `.venv`, где завершился v42. Это проверка качества, не обучение.

Зафиксирован R03 / step400 / 12800 предъявлений. Меняется только первый R1 в ranking; исходная ветка кандидатов/отказов и порог v25 сохраняются. Поэтому нужны 5 энкодеров, а не 4; банк признаков кандидата — 2560 float32.

Только исходная validation: 309 query и 896 gallery. Ни подбор на validation, ни calibration-поиск, ни новое обучение, ни автоматическое обновление MVP не выполняются. Эта validation уже наблюдалась раньше, это не независимый скрытый test.


In [1]:
import os, sys, subprocess
from pathlib import Path

os.environ['ORT_DISABLE_TELEMETRY'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '0'
os.environ['PYTORCH_MPS_FAST_MATH'] = '0'
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'training/osnet_r1_validation.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook в основном Car-classification-MSK, не в V41_Mac_M4')
if sys.prefix == sys.base_prefix:
    raise RuntimeError('Выбери kernel проектной .venv, не System Python')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_NAME = 'validation_v1'
DEVICE = 'mps'  # явный MPS, без скрытого CPU fallback
print('Python:', sys.executable, '\nПроект:', ROOT, '\nRun:', RUN_NAME, '\nDevice:', DEVICE)


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Проект: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK 
Run: validation_v1 
Device: mps


In [2]:
print('ЭТАП 1/3 — проверки scorer, отказов, экспорта и изоляции query')
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_osnet_r1_validation.py'], check=True)


ЭТАП 1/3 — проверки scorer, отказов, экспорта и изоляции query
..................                                                       [100%]
18 passed in 2.24s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_osnet_r1_validation.py'], returncode=0)

In [3]:
print('ЭТАП 2/3 — preflight, проверка весов, извлечение R03 и сравнение с v25')
from training.osnet_r1_validation import run, VARIANT
results = run(RUN_NAME, DEVICE)


ЭТАП 2/3 — preflight, проверка весов, извлечение R03 и сравнение с v25
PREFLIGHT OK: 309 query / 896 gallery; R03 step400; mps; no training
STAGE runtime_probe: start
FEATURES 16/32
FEATURES 32/32
Dual-role CPU: 8/8 images
STAGE runtime_probe: done in 1.9s
STAGE features: start
FEATURES 16/1205
FEATURES 272/1205
FEATURES 528/1205
FEATURES 784/1205
FEATURES 1040/1205
FEATURES 1205/1205
STAGE features: done in 16.7s
STAGE MVP_fusion_v25: start
STAGE MVP_fusion_v25: done in 0.9s
STAGE R03_replace_ranking_only: start
STAGE R03_replace_ranking_only: done in 1.3s
ARCHIVE: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_43_r03_validation/validation_v1_analysis.zip (0.5 MiB)
DONE: delta mAP=-0.009822; MVP unchanged


In [4]:
from IPython.display import Markdown, display
print('ЭТАП 3/3 — итоговый отчёт')
display(Markdown((VARIANT / 'runs' / RUN_NAME / 'REPORT.md').read_text()))
print('Архив для анализа:', VARIANT / f'{RUN_NAME}_analysis.zip')
print('MVP не изменён. После завершения сообщи: v43 завершился.')


ЭТАП 3/3 — итоговый отчёт


# v43 — сохранённый R03 против v25

Завершено; нового обучения нет.

| Система | mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|
| MVP_fusion_v25 | 0.828957 | 0.829960 | 0.795455 | 0.709677 |
| R03_replace_ranking_only | 0.819135 | 0.817814 | 0.795455 | 0.709677 |

Δ mAP: -0.009822.
Запросов лучше/хуже/без изменения AP: 15/19/213.
Парный bootstrap 95%: [-0.021942010666164802, 0.0010495662615023506] (описательная оценка).

R03 step400 / 12800 предъявлений; другие checkpoints на validation не проверялись.
Кандидаты/отказы побайтно совпали с v25; threshold = 0.534365177154541.
Ranking: MVP=.5; R03/R1_seed16/R1_seed17 по 1/6; k1=20, k2=3, lambda=.5.
Сохранение исходной кандидатской ветки требует 5 энкодеров против 4 у v25.
Вектор кандидата: реальные 2560 float32 координат с отдельным старым R1-блоком.
Исходная validation ранее уже использовалась; это не новый независимый test.
ONNX-export R03, скорость и внедрение в приложение в этот этап не входят.
MVP, v41, v42, bbox, исходные данные и веса не изменены. Автопродвижения нет.


Архив для анализа: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_43_r03_validation/validation_v1_analysis.zip
MVP не изменён. После завершения сообщи: v43 завершился.


### Продолжение и ограничения

При прерывании: Restart Kernel → Run All с прежним RUN_NAME. Завершённые этапы проверяются по SHA256; их повторное использование обозначается как cache. Изменившиеся источники/веса — ошибка, не повод увеличивать допуски.

Нужен основной проект с `dataset/`, историческим v25, завершённым v42, `research_transfer/v41_inputs/inputs.json` и соседней актуальной `Car-classification-MSK-main`. NiVe-изображения и `v41_inputs.zip` не используются. Новых скачиваний нет. Свободное место ≥1 GiB. Лимита времени нет.

Архив включает отчёты и per-query результаты, но не исходные веса или матрицы признаков. Полные exports и NPY сохраняются в новом `runs/validation_v1`. Ничего из прежних runs удалять/заменять не нужно. ONNX-поставка и скорость будущей пятиэнкодерной системы — отдельный этап после оценки качества.
